In [ ]:
import sys
import torch
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import numpy as np

from sklearn.datasets import make_moons

sys.path.extend(["../"])

from flow_models import NICEVP, NICE, RealNVP

torch.manual_seed(1)

In [ ]:
display(Markdown(open("../_macros.md").read()))

# Flow Models

Flow models are generative models that map a latent distribution $p(\zvec)$ to an observed distribution $p(\xvec)$ through a function $f(\xvec)$. 

This is very similar to what a probabilistic graphical model does. However, the main difference relies on $f()$ being a bijection. This can be easily achieved through these two restrictions:

* Both $\zvec$ and $\xvec$ have same dimensionality.
* $f()$ is invertible

The idea is to define an invertible mapping $f(\xvec)$ such that:

$$
\begin{align*}
\zvec = f(\xvec)\\
\xvec = f^{-1}(\zvec)
\end{align*}
$$

Alongside a prior probability over $\zvec$, $p(\zvec)$.


Under this setting, we can write the probability $p(\xvec)$ as:

$$
\begin{align*}
p(\xvec) &= p(\zvec)\mid\det \frac{\text{d}f(\xvec)}{\text{d} \xvec}\mid\\
&=p(f(\xvec))\mid\det \frac{\text{d}f(\xvec)}{\text{d} \xvec}\mid
\end{align*}
$$

There is a theorem out there which allows us to write the above formula as:

$$
\begin{align*}
p(\xvec) &= p(\zvec)\mid\det \frac{\text{d}f^{-1}(\zvec)}{\text{d} \zvec}\mid^{-1}
\end{align*}
$$

Which of the directions to use depends strongly on the application (variational inference, generative modelling) and whether the direct or inverse mapping is easy to invert. Also, one can find the direct mapping defined as $\xvec = f(\zvec)$, and so we need to plug in the change in the equations above.

One appealing property of such invertible transformations is that we can construct arbitrary complex $f()$ by a composition of elementary invertible functions such that:

$$
f(\xvec) = f_K\circ f_{K-1} \circ \dots \circ f_1(\xvec)
$$


If this is the case, then we have:

$$
p(\xvec) = p(f(\xvec))\prod^{K}_{k=1}\mid\det \frac{\text{d}f_k(\xvec_{k-1})}{\text{d} \xvec_{k-1}}\mid
$$

where $\mid \det \frac{\text{d}f_k(\xvec_{k-1})}{\xvec_{k-1}}\mid$ is the absolute value of the determinant of the Jacobian of the transformation $f_k$. Alternatively, we have:

$$
p(\xvec) = p(f(\xvec))\prod^{K}_{k=1}\mid\det \frac{\text{d}f^{-1}_k(\xvec_{k})}{\text{d} \xvec_{k}}\mid^{-1}
$$


## Training of such models

Since $p(\xvec)$ has an explicit formula, we can maximize the log probability over some dataset, such that:

$$
\begin{align*}
\log \prod^N_{n=1} p(\xvec) &= \sum^N_{n=1} \log p(f(\xvec^{(n)}))\prod^{K}_{k=1}\mid\det \frac{\text{d}f_k(\xvec^{(n)}_{k-1})}{\text{d} \xvec^{(n)}_{k-1}}\mid\\
&= \sum^N_{n=1}\log p(f(\xvec^{(n)})) + \sum^N_{n=1}\sum^{K}_{k=1}\log \mid\det \frac{\text{d}f_k(\xvec^{(n)}_{k-1})}{\text{d} \xvec^{(n)}_{k-1}}\mid
\end{align*}
$$

Thus, the main bottleneck is computing the determinant of the Jacobian, which typically requires a cubic cost. However, if we can construct transformations such that the Jacobian is diagonal or lower/upper triangular, then the log determinant is just the sum of the diagonals. Well, flow generative modelling is concerned exactly with that. Note that diagonal Jacobians are not of interest since that would require that we are only able to model the distribution over $\xvec$ by changing the marginals from $p(\zvec)$. In other words, the dependencies are exactly modelled by the same copula as the base distribution $p(\zvec)$.

On the other hand, when the determinant of the Jacobian is $1$, these transformations are known as volume-preserving, which means that a region of density in $\zvec$ is exactly mapped to a region of density in $\xvec$. This implies that the transformation $f()$ cannot contract or expand the region over which some mass of probability is assigned.

When the transformation is volume-preserving, we just need to optimize:

$$
\begin{align*}
\log \prod^N_{n=1} p(\xvec) &= \sum^N_{n=1} \log p(f(\xvec^{(n)}))\prod^{K}_{k=1}\mid\det \frac{\text{d}f_k(\xvec^{(n)}_{k-1})}{\text{d} \xvec^{(n)}_{k-1}}\mid\\
&= \sum^N_{n=1}\log p(f(\xvec^{(n)})) 
\end{align*}
$$

## NICE

### Volume Preserving Version

https://arxiv.org/pdf/1410.8516

Non-linear independent component estimation is a way of constructing $f()$ such that we can start from a simple prior independent latent distribution and use $f()$ to construct joint dependent distributions. 

This results in a lower-triangular Jacobian, which is volume-preserving, since the diagonal entries are 1.

Assume we have data with dimensionality $D$, i.e., $\xvec\in\mathbb{R}^D$. Assume we partition $\xvec$ the first $D/2$ dimensions and the rest such that:

$$
\begin{align*}
\xvec_a = \xvec_{1\dots D/2}\\
\xvec_b = \xvec_{D/2+1\dots D}
\end{align*}
$$

Each of the steps in $f()$ is constructed either by:

$$
\begin{align*}
\yvec_a &= \xvec_a\\
\yvec_b &= \xvec_b + \text{NNET}(\xvec_a)
\end{align*}
$$

or

$$
\begin{align*}
\yvec_a &= \xvec_a + \text{NNET}(\xvec_b)\\
\yvec_b &= \xvec_b
\end{align*}
$$

such that all dimensions influence all dimensions; otherwise, some of the dimensions will just be copies of the input. Each step in this transformation has an inverse given by:

$$
\begin{align*}
\xvec_a &= \yvec_a\\
\xvec_b &= \yvec_b - \text{NNET}(\xvec_a)
\end{align*}
$$

The Jacobian of the transformation is given by:

$$
\begin{align*}
\begin{pmatrix}
\frac{\text{d} \yvec_a}{\text{d} \xvec_a} & \frac{\text{d}\yvec_a}{\text{d}\xvec_b}\\
\frac{\text{d}\yvec_b}{\text{d} \xvec_a} & \frac{\text{d}\yvec_b}{\text{d}\xvec_b}\\
\end{pmatrix}=
%
\begin{pmatrix}
\Imat & 0\\
\frac{\text{d}\yvec_b}{\text{d} \xvec_a} & \Imat\\
\end{pmatrix}
\end{align*}
$$

and so the transformation has a unit Jacobian (product of diagonals).

### Non-Volume-Preserving Version

In the work, since the Jacobian of the affine coupling layer is 1, this implies that the flow cannot change the amount of assigned density between the latent and observed space. 

To overcome this limitation, the authors append a final layer consisting of a diagonal matrix that scales the input. Such a transformation is given by:

$$
\yvec = \diag\pare{\svec} \xvec
$$

where $\svec$ is the vector containing the per-dimension scaling. The Jacobian of such a transformation is given by:


$$
\begin{align*}
\begin{pmatrix}
\frac{\partial y_1}{\partial x_1} & \frac{\partial y_1}{\partial x_2} & \cdots & \frac{\partial y_1}{\partial x_D} \\
\frac{\partial y_2}{\partial x_1} & \frac{\partial y_2}{\partial x_2} & \cdots & \frac{\partial y_2}{\partial x_D} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{\partial y_D}{\partial x_1} & \frac{\partial y_D}{\partial x_2} & \cdots & \frac{\partial y_D}{\partial x_D}
\end{pmatrix}=
%
\begin{pmatrix}
s_1 & 0 & \cdots & 0 \\
0 & s_2 & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & s_D
\end{pmatrix}
\end{align*}
$$

In this case, the log-determinant of the Jacobian is given by the sum of the logarithms of the absolute values of the diagonal.

## Real Valued Non-Volume-Preserving Flow

https://arxiv.org/abs/1605.08803

This is an extension of the NICE work. In this case, the coupling layers are defined as:

$$
\begin{align*}
\xvec_a = \xvec_{1\dots D/2}\\
\xvec_b = \xvec_{D/2+1\dots D}
\end{align*}
$$

Each of the steps in $f()$ is constructed either by:

$$
\begin{align*}
\yvec_a &= \xvec_a\\
\yvec_b &= \xvec_b \circ e^{\text{NNET}_1(\xvec_a)} + \text{NNET}_2(\xvec_a)
\end{align*}
$$

$$
\begin{align*}
\yvec_a &= \xvec_a \circ e^{\text{NNET}_1(\xvec_b)} + \text{NNET}_2(\xvec_b)\\
\yvec_b &= \xvec_b
\end{align*}
$$

Each step in this transformation has an inverse given by:

$$
\begin{align*}
\xvec_a &= \yvec_a\\
\xvec_b &= e^{-\text{NNET}_1(\xvec_a)}\circ(\yvec_b - \text{NNET}_2(\xvec_a))
\end{align*}
$$

The Jacobian of the transformation is given by:

$$
\begin{align*}
\begin{pmatrix}
\frac{\text{d} \yvec_a}{\text{d} \xvec_a} & \frac{\text{d}\yvec_a}{\text{d}\xvec_b}\\
\frac{\text{d}\yvec_b}{\text{d} \xvec_a} & \frac{\text{d}\yvec_b}{\text{d}\xvec_b}\\
\end{pmatrix}=
%
\begin{pmatrix}
\Imat & 0\\
\frac{\text{d}\yvec_b}{\text{d} \xvec_a} & \text{diag}(e^{\text{NNET}_1(\xvec_a)})
\end{pmatrix}
\end{align*}
$$

This transformation is non-volume-preserving, as seen by the Jacobian. Since it is lower triangular, the Jacobian is the product of the diagonals. As such, the log-determinant Jacobian of this transformation is:

$$
\begin{align*}
\sum_{d=D/2+1}^{D} \log \exp\pare{[\text{NNET}_1(\xvec_a)]_d} \\
\sum_{d=D/2+1}^{D} \bra{\text{NNET}_1(\xvec_a)}_d
\end{align*}
$$


### Generate Data

In [ ]:
# -----------------------------
# Generar dataset Two Moons
# -----------------------------
X, _ = make_moons(n_samples=2000, noise=0.1, random_state=42)

plt.plot(X[:,0],X[:,1],'o', color = "C0", markersize = 2)

### Create torch tensor dataset

In [ ]:
X = torch.tensor(X, dtype=torch.float32)
dataset = TensorDataset(X)

batch_size = 2000
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

### Train Model Volume Preserving NICE

In [ ]:
model = NICEVP(dim = 2, hidden_dim=128, num_coupling_layers = 6)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 3000

for epoch in range(epochs):
    for x_tr, in dataloader:
        loss = -model.log_likelihood(x_tr).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 100 == 0:
        print(f"Epoch {epoch+1}, Loss: {-loss.item():.4f}")

In [ ]:
# -----------------------------
# Muestreo
# -----------------------------
torch.manual_seed(2)
with torch.no_grad():
    samples, prior_samples = model.generate_sample(1000)
    z_given_x = model.inference(X)

# -----------------------------
# Visualización
# -----------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8)) 

grid_size = 30
x = np.linspace(-4, 4, grid_size)
y = np.linspace(-4, 4, grid_size)
xx, yy = np.meshgrid(x, y)
points_plot = torch.tensor(np.stack([xx.flatten(), yy.flatten()], axis=1), dtype=torch.float32)

with torch.no_grad():
    z_space = model.x_to_z(points_plot)
    x_space = model.z_to_x(points_plot)

# original dataset
axes[0, 0].plot(X[:, 0], X[:, 1], 'o', markersize=0.5, zorder = 10)
axes[0, 0].set_title("Original Two Moons")

# original dataset mapped to z
axes[0, 1].plot(z_given_x[:, 0], z_given_x[:, 1], 'o', markersize=0.5, zorder = 10)
axes[0, 1].set_title("Latent space on Two Moons")

# samples from p(z)
axes[1, 0].plot(prior_samples[:, 0], prior_samples[:, 1], 'o', markersize=3, color='orange')
axes[1, 0].set_title("Samples from prior")

# samples from the flow
axes[1, 1].plot(samples[:, 0], samples[:, 1], 'o', markersize=3, color='orange')
axes[1, 1].set_title("Samples from Volume Preserving NICE")

### Training a Non-Volume Preserving NICE

In [ ]:
model = NICE(dim = 2, hidden_dim=128, num_coupling_layers = 6)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 3000

for epoch in range(epochs):
    for x_tr, in dataloader:
        loss = -model.log_likelihood(x_tr).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 100 == 0:
        print(f"Epoch {epoch+1}, Loss: {-loss.item():.4f}")

In [ ]:
# -----------------------------
# Muestreo
# -----------------------------
torch.manual_seed(2)
with torch.no_grad():
    samples, prior_samples = model.generate_sample(1000)
    z_given_x = model.inference(X)

# -----------------------------
# Visualización
# -----------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8)) 

grid_size = 30
x = np.linspace(-4, 4, grid_size)
y = np.linspace(-4, 4, grid_size)
xx, yy = np.meshgrid(x, y)
points_plot = torch.tensor(np.stack([xx.flatten(), yy.flatten()], axis=1), dtype=torch.float32)

with torch.no_grad():
    z_space = model.x_to_z(points_plot)
    x_space = model.z_to_x(points_plot)

# original dataset
axes[0, 0].plot(X[:, 0], X[:, 1], 'o', markersize=0.5, zorder = 10)
axes[0, 0].set_title("Original Two Moons")

# original dataset mapped to z
axes[0, 1].plot(z_given_x[:, 0], z_given_x[:, 1], 'o', markersize=0.5, zorder = 10)
axes[0, 1].set_title("Latent space on Two Moons")

# samples from p(z)
axes[1, 0].plot(prior_samples[:, 0], prior_samples[:, 1], 'o', markersize=3, color='orange')
axes[1, 0].set_title("Samples from prior")

# samples from the flow
axes[1, 1].plot(samples[:, 0], samples[:, 1], 'o', markersize=3, color='orange')
axes[1, 1].set_title("Samples from Real NVP")

### Training a Real NVP

In [ ]:
model = RealNVP(dim = 2, hidden_dim=128, num_coupling_layers = 6)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 3000

for epoch in range(epochs):
    for x_tr, in dataloader:
        loss = -model.log_likelihood(x_tr).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 100 == 0:
        print(f"Epoch {epoch+1}, Loss: {-loss.item():.4f}")

In [ ]:
# -----------------------------
# Muestreo
# -----------------------------
torch.manual_seed(2)
with torch.no_grad():
    samples, prior_samples = model.generate_sample(1000)
    z_given_x = model.inference(X)

# -----------------------------
# Visualización
# -----------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8)) 

grid_size = 30
x = np.linspace(-4, 4, grid_size)
y = np.linspace(-4, 4, grid_size)
xx, yy = np.meshgrid(x, y)
points_plot = torch.tensor(np.stack([xx.flatten(), yy.flatten()], axis=1), dtype=torch.float32)

with torch.no_grad():
    z_space = model.x_to_z(points_plot)
    x_space = model.z_to_x(points_plot)

# original dataset
axes[0, 0].plot(X[:, 0], X[:, 1], 'o', markersize=0.5, zorder = 10)
axes[0, 0].set_title("Original Two Moons")

# original dataset mapped to z
axes[0, 1].plot(z_given_x[:, 0], z_given_x[:, 1], 'o', markersize=0.5, zorder = 10)
axes[0, 1].set_title("Latent space on Two Moons")

# samples from p(z)
axes[1, 0].plot(prior_samples[:, 0], prior_samples[:, 1], 'o', markersize=3, color='orange')
axes[1, 0].set_title("Samples from prior")

# samples from the flow
axes[1, 1].plot(samples[:, 0], samples[:, 1], 'o', markersize=3, color='orange')
axes[1, 1].set_title("Samples from Real NVP")

### How the space is deformed by the model?

In [ ]:
model = RealNVP(dim = 2, hidden_dim=10, num_coupling_layers =10)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 1500

for epoch in range(epochs):
    for x_tr, in dataloader:
        loss = -model.log_likelihood(x_tr).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 100 == 0:
        print(f"Epoch {epoch+1}, Loss: {-loss.item():.4f}")

In [ ]:
def plot_grid(points, ax, grid_size, color='gray', alpha=1.0, linewidth=0.5):
    # Convertir a numpy si es tensor
    points_np = points.detach().cpu().numpy() if hasattr(points, "detach") else points
    # Reshape a (grid_size, grid_size, 2)
    grid = points_np.reshape(grid_size, grid_size, 2)

    # Dibujar líneas horizontales
    for i in range(grid_size):
        ax.plot(grid[i, :, 0], grid[i, :, 1], color=color, alpha=alpha, linewidth=linewidth)
    # Dibujar líneas verticales
    for j in range(grid_size):
        ax.plot(grid[:, j, 0], grid[:, j, 1], color=color, alpha=alpha, linewidth=linewidth)

# -----------------------------
# Muestreo
# -----------------------------
torch.manual_seed(2)
with torch.no_grad():
    samples, prior_samples = model.generate_sample(1000)
    z_given_x = model.inference(X)

# -----------------------------
# Visualización
# -----------------------------
fig, axes = plt.subplots(2, 3, figsize=(12, 8)) 

## define grid to plot deformation
grid_size_z_space = 20
x = np.linspace(-8, 8, grid_size_z_space)
y = np.linspace(-8, 8, grid_size_z_space)
xx, yy = np.meshgrid(x, y)
points_zspace = torch.tensor(np.stack([xx.flatten(), yy.flatten()], axis=1), dtype=torch.float32)

grid_size_x_space = 10
x = np.linspace(-12, 12, grid_size_x_space)
y = np.linspace(-12, 12, grid_size_x_space)
xx, yy = np.meshgrid(x, y)
points_xspace = torch.tensor(np.stack([xx.flatten(), yy.flatten()], axis=1), dtype=torch.float32)

gs_untransformed = 30
x = np.linspace(-4, 4, gs_untransformed)
y = np.linspace(-4, 4, gs_untransformed)
xx, yy = np.meshgrid(x, y)
points_plot_untransformed = torch.tensor(np.stack([xx.flatten(), yy.flatten()], axis=1), dtype=torch.float32)

with torch.no_grad():
    z_space = model.x_to_z(points_zspace)
    x_space = model.z_to_x(points_xspace)

# Primer subplot
axes[0, 0].plot(X[:, 0], X[:, 1], 'o', markersize=0.5, zorder = 10)
axes[0, 0].set_title("Original Two Moons")
plot_grid(points_plot_untransformed, axes[0, 0], grid_size=gs_untransformed)  # rejilla original sobre datos
axes[0, 0].set_xlim([-2,2.5])
axes[0, 0].set_ylim([-1.5,1.5])

# Segundo subplot
axes[0, 1].plot(z_given_x[:, 0], z_given_x[:, 1], 'o', markersize=0.5, zorder = 10)
axes[0, 1].set_title("Latent space on Two Moons")
plot_grid(z_space, axes[0, 1], grid_size_z_space)  # rejilla deformada
axes[0, 1].set_xlim([-4.3,4.3])
axes[0, 1].set_ylim([-4.3,4.3])

# Tercer subplot
axes[0, 2].plot(z_given_x[:, 0], z_given_x[:, 1], 'o', markersize=0.5, zorder = 10)
axes[0, 2].set_title("Latent space on Two Moons")
plot_grid(z_space, axes[0, 2], grid_size_z_space)  # rejilla deformada

# cuarto subplot
axes[1, 0].plot(prior_samples[:, 0], prior_samples[:, 1], 'o', markersize=3, color='orange')
axes[1, 0].set_title("Samples from prior")
plot_grid(points_plot_untransformed, axes[1, 0], gs_untransformed)  # rejilla original sobre datos
# axes[1, 0].set_xlim([-2,2.5])
# axes[1, 0].set_ylim([-1.5,1.5])

# quinto subplot
axes[1, 1].plot(samples[:, 0], samples[:, 1], 'o', markersize=3, color='orange')
axes[1, 1].set_title("Samples from Real NVP")
plot_grid(x_space, axes[1, 1], grid_size_x_space)  # rejilla deformada
axes[1, 1].set_xlim([-2.,3])
axes[1, 1].set_ylim([-1.5,1.5])

# sexto subplot
axes[1, 2].plot(samples[:, 0], samples[:, 1], 'o', markersize=0.5, zorder = 10)
axes[1, 2].set_title("Samples from Real NVP")
plot_grid(x_space, axes[1, 2], grid_size_x_space)  # rejilla deformada